# Stage 1 — 09A: A7 hard-slice & temporal-uncertainty audit

This notebook **does not train**. It starts from the original DLC-only A7
`V-JEPA 2.1-B` checkpoint and asks a different question from notebook 05:

> If 1/3/5-clip mean aggregation is already saturated on the 104-video DLC val,
> which videos / metadata slices are fragile even when their final label is correct?

Notebook 05 already showed that 1/3/5 deterministic clips all reach 1.0 on the
fixed validation split. 09A therefore runs **five clips once**, keeps clip-level
probabilities, and analyses centre/mean/median/trimmed/logit/max/top-2
aggregation, probability margin, temporal variance/range, clip disagreement,
and device/condition/document-type/group slices.

Final A7-compatible evaluation here uses **FP32 with no autocast**, matching the
safe submission path after the earlier attention dtype issue.


## 1. Setup


In [1]:
from __future__ import annotations

import copy
import gc
import json
import math
import os
import subprocess
import sys
import time
from pathlib import Path

REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage1-sangchun"

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)
except ModuleNotFoundError:
    print("Not running in Google Colab; Drive mount skipped.")

if IN_COLAB:
    REPO_ROOT = Path("/content/Blackbox-Detection")
    if not (REPO_ROOT / ".git").is_dir():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", BRANCH,
            "--single-branch", REPO_URL, str(REPO_ROOT),
        ], check=True)
    else:
        current_branch = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "branch", "--show-current"],
            check=True, capture_output=True, text=True,
        ).stdout.strip()
        if current_branch != BRANCH:
            subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", BRANCH], check=True)
        dirty = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "status", "--porcelain"],
            check=True, capture_output=True, text=True,
        ).stdout.strip()
        if dirty:
            print("WARNING: local repo has changes; git pull skipped.")
        else:
            subprocess.run(
                ["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", BRANCH],
                check=True,
            )
else:
    REPO_ROOT = Path.cwd().resolve()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / "pyproject.toml").is_file():
        raise FileNotFoundError("Run this notebook inside Blackbox-Detection repository.")

os.chdir(REPO_ROOT)

# Keep Colab's binary scientific stack intact. Install only the Stage 1 extras
# and then this repository editable with --no-deps, matching the current notebooks.
COLAB_EXTRAS = [
    "av>=15,<17", "timm==1.0.15", "fvcore==0.1.5.post20221221",
    "iopath==0.1.10", "yacs==0.1.8", "einops==0.8.1",
    "omegaconf==2.3.0", "hydra-core==1.3.2", "easydict==1.13",
]
if IN_COLAB:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed", *COLAB_EXTRAS,
    ], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_ROOT)
], check=True)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
import torch
import yaml

from blackbox_detection.utils import load_checkpoint, seed_everything

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATASET_ROOT = DRIVE_PROJECT_ROOT / "DATASET"
DLC_ROOT = DATASET_ROOT / "DLC-2021"
DLC_SPLIT_CSV = DLC_ROOT / "dlc_split.csv"
OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "outputs" / "stage1"
CONFIG_DIR = REPO_ROOT / "configs" / "stage1"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    required = {"DLC_ROOT": DLC_ROOT, "DLC_SPLIT_CSV": DLC_SPLIT_CSV}
    missing = [f"{k}: {v}" for k, v in required.items() if not v.exists()]
    if missing:
        raise FileNotFoundError("Missing required Drive paths:\n  " + "\n  ".join(missing))

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print("repo   :", REPO_ROOT)
print("branch :", BRANCH)
print("commit :", GIT_COMMIT)
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu    :", torch.cuda.get_device_name(0))


Mounted at /content/drive
repo   : /content/Blackbox-Detection
branch : stage1-sangchun
commit : 033cc92
torch  : 2.8.0+cu126
cuda   : True
gpu    : NVIDIA L4


## 2. Load exact A7 V-JEPA 2.1-B configuration


In [2]:
from blackbox_detection.stage1.dataset import Stage1VideoDataset, build_dataloader, video_batch_adapter
from blackbox_detection.stage1.evaluator import (
    AggregationConfig, Stage1Evaluator, aggregate_unit_predictions,
    evaluate_predictions, probabilities_to_labels, save_predictions,
)
from blackbox_detection.stage1.models import build_stage1_model, count_parameters
from blackbox_detection.stage1.sampling import build_clip_sampler
from blackbox_detection.stage1.transforms import ClipAugmentConfig, build_video_transforms

MODEL_NAME = "vjepa2_1_b"
CONFIG = yaml.safe_load((CONFIG_DIR / "vjepa2_1_b.yaml").read_text(encoding="utf-8"))
assert CONFIG["model"]["name"] == MODEL_NAME
SEED = int(CONFIG["train"]["seed"])
seed_everything(SEED, deterministic=False)

VJEPA_SOURCE_ROOT = Path("/content/vjepa2")
VJEPA_CKPT_DIR = DRIVE_PROJECT_ROOT / "pretrained" / "vjepa2"
VJEPA_CKPT_DIR.mkdir(parents=True, exist_ok=True)
VJEPA_CKPT = VJEPA_CKPT_DIR / "vjepa2_1_vitb_dist_vitG_384.pt"
if not (VJEPA_SOURCE_ROOT / ".git").is_dir():
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/facebookresearch/vjepa2.git",
        str(VJEPA_SOURCE_ROOT),
    ], check=True)
if not VJEPA_CKPT.is_file():
    subprocess.run([
        "wget", "-O", str(VJEPA_CKPT),
        "https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt",
    ], check=True)
params = dict(CONFIG["model"]["params"])
params["source_root"] = str(VJEPA_SOURCE_ROOT)
params["checkpoint_path"] = str(VJEPA_CKPT)
params["allow_download"] = False
print("V-JEPA source :", params["source_root"])
print("V-JEPA ckpt   :", params["checkpoint_path"])
print("arch          :", params["arch"])
print("runtime input :", params["input_frames"], "frames @", params["input_size"])


V-JEPA source : /content/vjepa2
V-JEPA ckpt   : /content/drive/MyDrive/Blackbox-Detection/pretrained/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt
arch          : vit_base
runtime input : 16 frames @ 384


## 3. Paths and fixed DLC validation manifest


In [3]:
A7_RUN_DIR = OUTPUT_ROOT / "dlc" / MODEL_NAME
A7_CKPT = A7_RUN_DIR / "best.pt"
RUN_DIR = OUTPUT_ROOT / "09a_a7_hard_validation"
RUN_DIR.mkdir(parents=True, exist_ok=True)
if not A7_CKPT.is_file():
    raise FileNotFoundError(f"A7 best.pt not found: {A7_CKPT}")
print("A7 checkpoint:", A7_CKPT)
print("09A output   :", RUN_DIR)


A7 checkpoint: /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc/vjepa2_1_b/best.pt
09A output   : /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/09a_a7_hard_validation


In [4]:
VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm"}

def _normalize_rel_text(value: str) -> str:
    return str(value).replace("\\", "/").strip().lstrip("./")

def _build_dlc_video_index() -> dict[tuple[str, str], str]:
    index = {}
    counts = {}
    for source in ("or", "re"):
        clips_root = DLC_ROOT / source / "clips_video"
        if not clips_root.is_dir():
            raise FileNotFoundError(f"DLC clips directory not found: {clips_root}")
        count = 0
        for path in clips_root.rglob("*"):
            if not path.is_file() or path.suffix.lower() not in VIDEO_EXTENSIONS:
                continue
            rel = path.relative_to(clips_root).with_suffix("").as_posix()
            key = (source, _normalize_rel_text(rel))
            if key in index and index[key] != str(path):
                raise ValueError(f"Duplicate DLC video key: {key}")
            index[key] = str(path)
            count += 1
        counts[source] = count
    print("indexed DLC videos:", counts)
    return index

DLC_VIDEO_INDEX = _build_dlc_video_index()

def _resolve_dlc_video(source: str, clip_id: str) -> str:
    source = str(source).strip().lower()
    clip_id = _normalize_rel_text(clip_id)
    key = (source, clip_id)
    if key in DLC_VIDEO_INDEX:
        return DLC_VIDEO_INDEX[key]
    matches = []
    for (src, rel), path in DLC_VIDEO_INDEX.items():
        if src != source:
            continue
        if rel.startswith(clip_id + "/") or rel.endswith("/" + clip_id) or rel == clip_id:
            matches.append(path)
    if len(matches) == 1:
        return matches[0]
    leaf = Path(clip_id).name
    matches = [
        path for (src, rel), path in DLC_VIDEO_INDEX.items()
        if src == source and Path(rel).name == leaf
    ]
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(f"Cannot resolve DLC source={source!r}, clip_id={clip_id!r}")

def load_dlc_manifest(split_name: str) -> pd.DataFrame:
    raw = pd.read_csv(DLC_SPLIT_CSV).copy()
    required = {
        "clip_id", "class", "source", "document_type", "document_id",
        "group", "split", "device", "condition",
    }
    missing = sorted(required - set(raw.columns))
    if missing:
        raise ValueError(f"dlc_split.csv missing columns: {missing}")
    raw["split"] = raw["split"].astype(str).str.strip().str.lower()
    raw["source"] = raw["source"].astype(str).str.strip().str.lower()
    raw["class"] = raw["class"].astype(str).str.strip().str.lower()
    frame = raw.loc[raw["split"].eq(split_name)].copy()
    if frame.empty:
        raise ValueError(f"No DLC rows for split={split_name!r}")
    frame["label"] = frame["class"].map({"original": "ORIGINAL", "rerecorded": "RERECORDED"})
    if frame["label"].isna().any():
        raise ValueError("Unexpected DLC class value found.")
    frame["video_id"] = "dlc__" + frame["clip_id"].astype(str).str.replace("/", "__", regex=False)
    frame["dataset"] = "dlc2021"
    frame["scene_type"] = "document"
    frame["video_path"] = [
        _resolve_dlc_video(source, clip_id)
        for source, clip_id in zip(frame["source"], frame["clip_id"])
    ]
    keep = [
        "video_path", "label", "video_id", "dataset", "scene_type",
        "clip_id", "source", "document_type", "document_id", "group",
        "device", "condition",
    ]
    out = frame[keep].reset_index(drop=True)
    missing_paths = [p for p in out["video_path"] if not Path(p).is_file()]
    if missing_paths:
        raise FileNotFoundError(
            f"{split_name}: {len(missing_paths)} missing videos; examples={missing_paths[:3]}"
        )
    return out


indexed DLC videos: {'or': 290, 're': 400}


In [5]:
val_df = load_dlc_manifest("val")
print("validation videos:", len(val_df))
display(pd.crosstab(val_df["label"], val_df["device"], margins=True))
display(pd.crosstab(val_df["label"], val_df["condition"], margins=True))


validation videos: 104


device,android,iphone,All
label,,,
ORIGINAL,27,17,44
RERECORDED,30,30,60
All,57,47,104


condition,artificial light,artificial light + fingers,artificial light + flash,colored light,colored light + flash,daylight,daylight + alternating shadow,frontally average range projectivity average,frontally close projectivity is average,frontally close without much projectivity,frontally medium range without special projectivity,hp,lenovo,low light,macbook pro,philips,All
label,,,,,,,,,,,,,,,,,
ORIGINAL,8,2,4,2,4,8,4,2,2,2,2,0,0,4,0,0,44
RERECORDED,0,0,0,0,0,0,0,0,0,0,0,6,6,0,24,24,60
All,8,2,4,2,4,8,4,2,2,2,2,6,6,4,24,24,104


## 4. Restore A7 and verify architecture


In [6]:
model = build_stage1_model(
    MODEL_NAME,
    finetune_mode=CONFIG["model"]["finetune_mode"],
    unfreeze_last_n=int(CONFIG["model"]["unfreeze_last_n"]),
    **params,
)
load_checkpoint(A7_CKPT, model=model, map_location="cpu", restore_rng_state=False)
print("blocks       :", len(model.blocks))
print("feature dim  :", model.feature_dim)
print("parameters   :", count_parameters(model))
print("preprocessing:", model.preprocessing())
print("load report  :", json.dumps(getattr(model, "load_report", {}), indent=2, default=str))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


blocks       : 12
feature dim  : 768
parameters   : {'total': 89199362, 'trainable': 2366210, 'frozen': 86833152}
preprocessing: {'input_kind': 'video', 'num_frames': 16, 'input_size': 384, 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'channels_first': True, 'source': 'https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt', 'notes': 'Official V-JEPA 2.1 preprocessing: shorter side resized to crop_size * 256 / 224, centre crop, ImageNet normalisation. The released 2.1 encoders are 384-resolution.'}
load report  : {
  "arch": "vit_base",
  "encoder_kwargs": {
    "patch_size": 16,
    "tubelet_size": 2,
    "use_sdpa": true,
    "use_SiLU": false,
    "wide_SiLU": true,
    "uniform_power": false,
    "use_rope": true,
    "img_temporal_dim_size": 1,
    "interpolate_rope": true,
    "img_size": [
      384,
      384
    ],
    "num_frames": 64
  },
  "weights": {
    "source": "/content/drive/MyDrive/Blackbox-Detection/pretrained/vjepa2/vjepa2_1_vitb_dist

## 5. Decode five deterministic clips once


In [7]:
video_cfg = CONFIG["data"]
aug_cfg = CONFIG["augmentation"]
pre = model.preprocessing()
_, val_transform = build_video_transforms(
    crop_size=int(pre["input_size"]),
    mean=tuple(pre["mean"]),
    std=tuple(pre["std"]),
    train_config=ClipAugmentConfig(
        crop_size=int(pre["input_size"]),
        scale_range=tuple(aug_cfg["scale_range"]),
        ratio_range=tuple(aug_cfg["ratio_range"]),
        hflip_prob=float(aug_cfg["hflip_prob"]),
        brightness=float(aug_cfg["brightness"]),
        contrast=float(aug_cfg["contrast"]),
        perspective_prob=float(aug_cfg["perspective_prob"]),
        perspective_scale=float(aug_cfg["perspective_scale"]),
    ),
)
NUM_CLIPS = 5
dataset = Stage1VideoDataset(
    val_df,
    clip_sampler=build_clip_sampler(
        train=False,
        num_frames=int(video_cfg["num_frames"]),
        val_stride=int(video_cfg["val_stride"]),
        num_clips=NUM_CLIPS,
    ),
    transform=val_transform,
    on_error="zero",
    deterministic=True,
)
loader = build_dataloader(
    dataset, batch_size=1, shuffle=False, num_workers=2, seed=SEED,
    persistent_workers=True,
)
evaluator = Stage1Evaluator(
    model, video_batch_adapter(), device=device, amp=False,
    aggregation=AggregationConfig(frame_method="mean", video_method="mean"),
)
started = time.perf_counter()
units = evaluator.predict_units(loader)
five_clip_elapsed = time.perf_counter() - started
units.to_csv(RUN_DIR / "a7_5clip_units.csv", index=False)
print(f"5-clip unit rows: {len(units)} | runtime: {five_clip_elapsed / 60:.2f} min")

# Exact A7 anchor: num_clips=1 uses DeterministicClipSampler's original
# max_start // 2 centre rule. The middle slot of a 5-clip sampler is usually
# close but is not guaranteed bit-identical for every odd max_start.
center_dataset = Stage1VideoDataset(
    val_df,
    clip_sampler=build_clip_sampler(
        train=False,
        num_frames=int(video_cfg["num_frames"]),
        val_stride=int(video_cfg["val_stride"]),
        num_clips=1,
    ),
    transform=val_transform,
    on_error="zero",
    deterministic=True,
)
center_loader = build_dataloader(
    center_dataset, batch_size=1, shuffle=False, num_workers=2, seed=SEED,
    persistent_workers=True,
)
center_started = time.perf_counter()
centre_units = evaluator.predict_units(center_loader)
center_elapsed = time.perf_counter() - center_started
centre_units.to_csv(RUN_DIR / "a7_center_units.csv", index=False)
elapsed = five_clip_elapsed + center_elapsed
print(f"exact centre rows: {len(centre_units)} | runtime: {center_elapsed / 60:.2f} min")
display(units.head(10))


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


5-clip unit rows: 520 | runtime: 11.35 min


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


exact centre rows: 104 | runtime: 1.94 min


,video_id,label,dataset,frame_index,patch_index,prob_rerecorded,valid
0,dlc__aze_passport__00.or0001,ORIGINAL,dlc2021,-1,0,0.000003,True
1,dlc__aze_passport__00.or0001,ORIGINAL,dlc2021,-1,1,0.000003,True
2,dlc__aze_passport__00.or0001,ORIGINAL,dlc2021,-1,2,0.000001,True
3,dlc__aze_passport__00.or0001,ORIGINAL,dlc2021,-1,3,0.000001,True
4,dlc__aze_passport__00.or0001,ORIGINAL,dlc2021,-1,4,0.000001,True
5,dlc__aze_passport__00.or0002,ORIGINAL,dlc2021,-1,0,0.000001,True
6,dlc__aze_passport__00.or0002,ORIGINAL,dlc2021,-1,1,0.000002,True
7,dlc__aze_passport__00.or0002,ORIGINAL,dlc2021,-1,2,0.000006,True
8,dlc__aze_passport__00.or0002,ORIGINAL,dlc2021,-1,3,0.000003,True
9,dlc__aze_passport__00.or0002,ORIGINAL,dlc2021,-1,4,0.000001,True


## 6. Centre clip and alternative aggregation


In [8]:
from blackbox_detection.utils.metrics import stage1_score

META_COLS = [
    "video_id", "clip_id", "source", "document_type", "document_id",
    "group", "device", "condition",
]
meta = val_df[META_COLS].drop_duplicates("video_id")

def _score_video_table(frame: pd.DataFrame, method: str):
    result = evaluate_predictions(frame, threshold=0.5, search_threshold=False)
    pred = result.predictions.merge(meta, on="video_id", how="left", validate="one_to_one")
    pred["method"] = method
    return result, pred

tables = {}
rows = []
centre_video = centre_units[["video_id", "label", "dataset", "prob_rerecorded"]].copy()
centre_video["prob_original"] = 1.0 - centre_video["prob_rerecorded"]
result, pred = _score_video_table(centre_video, "center")
tables["center"] = pred
rows.append({"method":"center", "macro_f1_at_0.5":float(result.macro_f1_at_default)})

for method in ("mean", "median", "trimmed_mean", "logit_mean", "max", "min"):
    video = aggregate_unit_predictions(
        units, aggregation=AggregationConfig(frame_method="mean", video_method=method)
    )
    result, pred = _score_video_table(video, method)
    tables[method] = pred
    rows.append({"method":method, "macro_f1_at_0.5":float(result.macro_f1_at_default)})

def topk_mean(values, k=2):
    values = np.sort(np.asarray(values, dtype=np.float64))
    return float(values[-min(k, len(values)):].mean())

top2 = (
    units.loc[units["valid"].astype(bool)]
    .groupby(["video_id", "label", "dataset"], as_index=False)["prob_rerecorded"]
    .agg(topk_mean)
)
top2["prob_original"] = 1.0 - top2["prob_rerecorded"]
result, pred = _score_video_table(top2, "top2_mean")
tables["top2_mean"] = pred
rows.append({"method":"top2_mean", "macro_f1_at_0.5":float(result.macro_f1_at_default)})
summary = pd.DataFrame(rows).sort_values(["macro_f1_at_0.5", "method"], ascending=[False, True])
summary.to_csv(RUN_DIR / "aggregation_summary.csv", index=False)
display(summary)


,method,macro_f1_at_0.5
0,center,1.00000
4,logit_mean,1.00000
5,max,1.00000
1,mean,1.00000
2,median,1.00000
7,top2_mean,1.00000
3,trimmed_mean,1.00000
6,min,0.99018


## 7. Temporal uncertainty and hard cases


In [9]:
valid_units = units.loc[units["valid"].astype(bool) & units["prob_rerecorded"].notna()].copy()
stats = (
    valid_units.groupby(["video_id", "label", "dataset"], as_index=False)
    .agg(
        p_mean=("prob_rerecorded", "mean"),
        p_std=("prob_rerecorded", "std"),
        p_min=("prob_rerecorded", "min"),
        p_max=("prob_rerecorded", "max"),
        p_median=("prob_rerecorded", "median"),
    )
)
stats["p_std"] = stats["p_std"].fillna(0.0)
stats["p_range"] = stats["p_max"] - stats["p_min"]
stats["margin"] = (stats["p_mean"] - 0.5).abs()
clip_labels = valid_units.assign(
    clip_pred=np.where(valid_units["prob_rerecorded"].ge(0.5), "RERECORDED", "ORIGINAL")
)
disagreement = clip_labels.groupby("video_id")["clip_pred"].nunique().rename("num_clip_labels").reset_index()
stats = stats.merge(disagreement, on="video_id", how="left")
stats["clip_label_disagreement"] = stats["num_clip_labels"].gt(1)
stats = stats.merge(meta, on="video_id", how="left", validate="one_to_one")
stats["rank_margin"] = stats["margin"].rank(method="min", ascending=True)
stats["rank_range"] = stats["p_range"].rank(method="min", ascending=False)
stats["hardness_score"] = stats["rank_margin"] + stats["rank_range"]
hard_cases = stats.sort_values(
    ["clip_label_disagreement", "hardness_score"], ascending=[False, True], kind="mergesort"
).reset_index(drop=True)
hard_cases.to_csv(RUN_DIR / "hard_cases.csv", index=False)
display(hard_cases.head(30))


,video_id,label,dataset,p_mean,p_std,p_min,p_max,p_median,p_range,margin,...,clip_id,source,document_type,document_id,group,device,condition,rank_margin,rank_range,hardness_score
0,dlc__svk_id__07.re0001,RERECORDED,dlc2021,0.844395,0.293699,0.320221,0.990583,0.984500,0.670362,0.344395,...,svk_id/07.re0001,re,svk_id,7,svk_id_07,iphone,macbook pro,2.0,1.0,3.0
1,dlc__svk_id__07.re0003,RERECORDED,dlc2021,0.659600,0.151245,0.504117,0.853896,0.659216,0.349780,0.159600,...,svk_id/07.re0003,re,svk_id,7,svk_id_07,android,macbook pro,1.0,2.0,3.0
2,dlc__svk_id__06.re0004,RERECORDED,dlc2021,0.992056,0.009629,0.975925,0.999052,0.997095,0.023126,0.492056,...,svk_id/06.re0004,re,svk_id,6,svk_id_06,android,philips,3.0,3.0,6.0
3,dlc__rus_internalpassport__01.re0006,RERECORDED,dlc2021,0.995379,0.009651,0.978119,0.999915,0.999580,0.021796,0.495379,...,rus_internalpassport/01.re0006,re,rus_internalpassport,1,rus_internalpassport_01,android,philips,4.0,4.0,8.0
4,dlc__grc_passport__05.or0003,ORIGINAL,dlc2021,0.004008,0.002980,0.001084,0.008393,0.003693,0.007309,0.495992,...,grc_passport/05.or0003,or,grc_passport,5,grc_passport_05,android,low light,6.0,5.0,11.0
5,dlc__svk_id__06.re0002,RERECORDED,dlc2021,0.995481,0.001676,0.993130,0.997813,0.995502,0.004683,0.495481,...,svk_id/06.re0002,re,svk_id,6,svk_id_06,iphone,philips,5.0,6.0,11.0
6,dlc__lva_passport__02.re0002,RERECORDED,dlc2021,0.997394,0.001097,0.996003,0.998880,0.997060,0.002877,0.497394,...,lva_passport/02.re0002,re,lva_passport,2,lva_passport_02,iphone,macbook pro,7.0,7.0,14.0
7,dlc__rus_internalpassport__01.re0003,RERECORDED,dlc2021,0.999051,0.000858,0.997581,0.999757,0.999391,0.002176,0.499051,...,rus_internalpassport/01.re0003,re,rus_internalpassport,1,rus_internalpassport_01,iphone,philips,10.0,8.0,18.0
8,dlc__svk_id__06.re0003,RERECORDED,dlc2021,0.998879,0.000546,0.998141,0.999614,0.998954,0.001473,0.498879,...,svk_id/06.re0003,re,svk_id,6,svk_id_06,android,macbook pro,9.0,9.0,18.0
9,dlc__lva_passport__02.re0005,RERECORDED,dlc2021,0.998832,0.000378,0.998435,0.999420,0.998676,0.000985,0.498832,...,lva_passport/02.re0005,re,lva_passport,2,lva_passport_02,android,macbook pro,8.0,13.0,21.0


## 8. Metadata slice audit


In [10]:
def slice_report(pred: pd.DataFrame, column: str, min_rows: int = 4) -> pd.DataFrame:
    if column in pred.columns:
        merged = pred.copy()
    else:
        merged = pred.merge(
            val_df[["video_id", column]].drop_duplicates("video_id"),
            on="video_id", how="left", validate="one_to_one",
        )
    out = []
    for value, group in merged.groupby(column, dropna=False):
        if len(group) < min_rows:
            continue
        payload = {
            "slice_column": column,
            "slice_value": str(value),
            "num_videos": int(len(group)),
            "num_original": int(group["label"].eq("ORIGINAL").sum()),
            "num_rerecorded": int(group["label"].eq("RERECORDED").sum()),
            "mean_margin": float((group["prob_rerecorded"] - 0.5).abs().mean()),
        }
        if group["label"].nunique() == 2:
            payload["macro_f1_at_0.5"] = float(stage1_score(
                group["label"], probabilities_to_labels(group["prob_rerecorded"], 0.5)
            ))
        else:
            payload["macro_f1_at_0.5"] = np.nan
        out.append(payload)
    return pd.DataFrame(out)

slice_frames = []
for column in ("device", "condition", "document_type", "group"):
    f = slice_report(tables["center"], column)
    if len(f):
        slice_frames.append(f)
slice_summary = pd.concat(slice_frames, ignore_index=True) if slice_frames else pd.DataFrame()
if len(slice_summary):
    slice_summary = slice_summary.sort_values(
        ["macro_f1_at_0.5", "mean_margin", "num_videos"],
        ascending=[True, True, False], na_position="last", kind="mergesort",
    )
slice_summary.to_csv(RUN_DIR / "slice_summary.csv", index=False)
display(slice_summary.head(50))


,slice_column,slice_value,num_videos,num_original,num_rerecorded,mean_margin,macro_f1_at_0.5
31,group,svk_id_07,6,2,4,0.473774,1.0
19,document_type,svk_id,14,6,8,0.486647,1.0
30,group,svk_id_06,8,4,4,0.496302,1.0
0,device,android,57,27,30,0.496727,1.0
24,group,grc_passport_05,10,6,4,0.499029,1.0
15,document_type,grc_passport,18,8,10,0.499402,1.0
26,group,lva_passport_02,8,2,6,0.499489,1.0
1,device,iphone,47,17,30,0.499530,1.0
17,document_type,rus_internalpassport,10,4,6,0.499763,1.0
28,group,rus_internalpassport_01,10,4,6,0.499763,1.0


## 9. Save A7 centre predictions and summary


In [11]:
a7_center = tables["center"].copy()
drop_cols = [c for c in META_COLS[1:] + ["method"] if c in a7_center.columns]
save_predictions(a7_center.drop(columns=drop_cols), RUN_DIR / "a7_center_val_predictions.csv")
audit = {
    "experiment": "09a_a7_hard_validation",
    "source_checkpoint": str(A7_CKPT),
    "git_commit": GIT_COMMIT,
    "num_clips": NUM_CLIPS,
    "fp32_no_autocast": True,
    "runtime_seconds": float(elapsed),
    "aggregation_summary": summary.to_dict(orient="records"),
    "num_temporal_label_disagreements": int(hard_cases["clip_label_disagreement"].sum()),
    "median_probability_range": float(hard_cases["p_range"].median()),
    "max_probability_range": float(hard_cases["p_range"].max()),
    "min_center_margin": float((a7_center["prob_rerecorded"] - 0.5).abs().min()),
}
(RUN_DIR / "summary.json").write_text(json.dumps(audit, indent=2, default=str), encoding="utf-8")
print(json.dumps(audit, indent=2))
print("saved to:", RUN_DIR)


{
  "experiment": "09a_a7_hard_validation",
  "source_checkpoint": "/content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc/vjepa2_1_b/best.pt",
  "git_commit": "033cc92",
  "num_clips": 5,
  "fp32_no_autocast": true,
  "runtime_seconds": 797.6509808159997,
  "aggregation_summary": [
    {
      "method": "center",
      "macro_f1_at_0.5": 1.0
    },
    {
      "method": "logit_mean",
      "macro_f1_at_0.5": 1.0
    },
    {
      "method": "max",
      "macro_f1_at_0.5": 1.0
    },
    {
      "method": "mean",
      "macro_f1_at_0.5": 1.0
    },
    {
      "method": "median",
      "macro_f1_at_0.5": 1.0
    },
    {
      "method": "top2_mean",
      "macro_f1_at_0.5": 1.0
    },
    {
      "method": "trimmed_mean",
      "macro_f1_at_0.5": 1.0
    },
    {
      "method": "min",
      "macro_f1_at_0.5": 0.9901803417996412
    }
  ],
  "num_temporal_label_disagreements": 1,
  "median_probability_range": 5.322694778442383e-05,
  "max_probability_range": 0.6703623533248901,
 

In [12]:
from pathlib import Path
import pandas as pd

RUN_DIR = Path(
    "/content/drive/MyDrive/Blackbox-Detection/"
    "outputs/stage1/09a_a7_hard_validation"
)

hard = pd.read_csv(
    RUN_DIR / "hard_cases.csv"
)

units = pd.read_csv(
    RUN_DIR / "a7_5clip_units.csv"
)

# temporal hard-label disagreement가 난 유일한 영상
unstable = hard.loc[
    hard["clip_label_disagreement"].astype(bool)
].copy()

print("=== TEMPORALLY UNSTABLE VIDEO ===")
display(
    unstable[
        [
            "video_id",
            "label",
            "clip_id",
            "document_type",
            "document_id",
            "group",
            "device",
            "condition",
            "p_mean",
            "p_min",
            "p_max",
            "p_range",
            "margin",
        ]
    ]
)

if len(unstable):
    vid = unstable.iloc[0]["video_id"]

    print("\n=== 5 CLIP PROBABILITIES ===")

    display(
        units.loc[
            units["video_id"].eq(vid),
            [
                "video_id",
                "patch_index",
                "prob_rerecorded",
                "valid",
            ],
        ].sort_values("patch_index")
    )

print("\n=== TOP 10 HARD CASES ===")

display(
    hard[
        [
            "video_id",
            "label",
            "document_type",
            "group",
            "device",
            "condition",
            "p_mean",
            "p_min",
            "p_max",
            "p_range",
            "margin",
        ]
    ].head(10)
)

=== TEMPORALLY UNSTABLE VIDEO ===


,video_id,label,clip_id,document_type,document_id,group,device,condition,p_mean,p_min,p_max,p_range,margin
0,dlc__svk_id__07.re0001,RERECORDED,svk_id/07.re0001,svk_id,7,svk_id_07,iphone,macbook pro,0.844395,0.320221,0.990583,0.670362,0.344395



=== 5 CLIP PROBABILITIES ===


,video_id,patch_index,prob_rerecorded,valid
500,dlc__svk_id__07.re0001,0,0.320221,True
501,dlc__svk_id__07.re0001,1,0.984500,True
502,dlc__svk_id__07.re0001,2,0.990583,True
503,dlc__svk_id__07.re0001,3,0.985503,True
504,dlc__svk_id__07.re0001,4,0.941168,True



=== TOP 10 HARD CASES ===


,video_id,label,document_type,group,device,condition,p_mean,p_min,p_max,p_range,margin
0,dlc__svk_id__07.re0001,RERECORDED,svk_id,svk_id_07,iphone,macbook pro,0.844395,0.320221,0.990583,0.670362,0.344395
1,dlc__svk_id__07.re0003,RERECORDED,svk_id,svk_id_07,android,macbook pro,0.659600,0.504117,0.853896,0.349780,0.159600
2,dlc__svk_id__06.re0004,RERECORDED,svk_id,svk_id_06,android,philips,0.992056,0.975925,0.999052,0.023126,0.492056
3,dlc__rus_internalpassport__01.re0006,RERECORDED,rus_internalpassport,rus_internalpassport_01,android,philips,0.995379,0.978119,0.999915,0.021796,0.495379
4,dlc__grc_passport__05.or0003,ORIGINAL,grc_passport,grc_passport_05,android,low light,0.004008,0.001084,0.008393,0.007309,0.495992
5,dlc__svk_id__06.re0002,RERECORDED,svk_id,svk_id_06,iphone,philips,0.995481,0.993130,0.997813,0.004683,0.495481
6,dlc__lva_passport__02.re0002,RERECORDED,lva_passport,lva_passport_02,iphone,macbook pro,0.997394,0.996003,0.998880,0.002877,0.497394
7,dlc__rus_internalpassport__01.re0003,RERECORDED,rus_internalpassport,rus_internalpassport_01,iphone,philips,0.999051,0.997581,0.999757,0.002176,0.499051
8,dlc__svk_id__06.re0003,RERECORDED,svk_id,svk_id_06,android,macbook pro,0.998879,0.998141,0.999614,0.001473,0.498879
9,dlc__lva_passport__02.re0005,RERECORDED,lva_passport,lva_passport_02,android,macbook pro,0.998832,0.998435,0.999420,0.000985,0.498832


## Interpretation

Do not pick an aggregation only because the same 104-video validation split
remains at 1.0. The useful outputs are the probability margins, temporal
instability, and metadata slices. They become stress-test diagnostics for
09B–11.
